<a href="https://colab.research.google.com/github/srivastava071/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

print("HF token found:", bool(HF_TOKEN))

Paste your Hugging Face READ token (hf_...): ··········
HF token found: True


In [ ]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("DuckDB connected.")

DuckDB connected.


In [ ]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
}

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srivastava071/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

I first look at the distributions of the signals I may use for the Refresh / Content Opportunity lane.

The main signals I will inspect are impressions, clicks, average position, and content age.

I expect search performance variables to be uneven, with many pages having relatively small values and a smaller number of pages having much larger values. I will use the observed distributions rather than assuming that a particular threshold is universally correct.

This check helps me understand the scale and possible heavy tails before defining any baseline rule.

In [ ]:
march = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()

print("March rows:", len(march))

print("\nDistribution summary:")
print(
    march[
        [
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position"
        ]
    ].describe()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March rows: 9841378

Distribution summary:
       gsc_impressions    gsc_clicks  gsc_avg_position
count     9.841378e+06  9.841378e+06      3.611061e+06
mean      2.851812e+01  8.350782e-02      1.582665e+01
std       1.559266e+02  7.814341e-01      1.985603e+01
min       0.000000e+00  0.000000e+00      0.000000e+00
25%       0.000000e+00  0.000000e+00      3.742120e+00
50%       0.000000e+00  0.000000e+00      7.500000e+00
75%       6.000000e+00  0.000000e+00      2.020000e+01
max       4.008400e+04  2.740000e+02      4.980000e+02


In [ ]:
for col in [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]:
    print(
        f"{col}: "
        f"zero = {(march[col] == 0).sum():,}, "
        f"missing = {march[col].isna().sum():,}"
    )

gsc_impressions: zero = 6,230,317, missing = 0
gsc_clicks: zero = 9,423,397, missing = 0
gsc_avg_position: zero = 163,189, missing = 6,230,317


### Observed result

The March data contains 9,841,378 rows. The distributions are strongly uneven and contain many zero values. Impressions have a median of 0 and a 75th percentile of 6, while the maximum is 40,084. Clicks are even sparser, with a median and 75th percentile of 0 and a maximum of 274.

Average position is available for fewer rows because 6,230,317 rows have missing position values. This matches the rows with zero impressions, so position should be interpreted only where search visibility exists.

These results show that simple averages or fixed thresholds should be used carefully. I will use bucketed comparisons and sample sizes when testing signals.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal test #1

In [ ]:
signal1 = con.sql(f"""
WITH content_30d AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_30d
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-02-01'
      AND report_date < DATE '2026-03-01'
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    CASE
        WHEN impressions_30d = 0 THEN '0'
        WHEN impressions_30d <= 10 THEN '1-10'
        WHEN impressions_30d <= 100 THEN '11-100'
        WHEN impressions_30d <= 1000 THEN '101-1000'
        ELSE '1000+'
    END AS impression_bucket,
    COUNT(*) AS n,
    AVG(impressions_30d) AS avg_impressions
FROM content_30d
GROUP BY 1
ORDER BY
    CASE impression_bucket
        WHEN '0' THEN 1
        WHEN '1-10' THEN 2
        WHEN '11-100' THEN 3
        WHEN '101-1000' THEN 4
        WHEN '1000+' THEN 5
    END
""").df()

signal1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impression_bucket,n,avg_impressions
0,0,167987,0.000000
1,1-10,35661,3.780741
2,11-100,37774,42.485016
3,101-1000,46837,387.622734
4,1000+,33287,4813.716526


In [ ]:
signal1_decline = con.sql(f"""
WITH monthly AS (
    SELECT
        client_hash_id,
        content_hash_id,
        DATE_TRUNC('month', report_date) AS month,
        SUM(gsc_impressions) AS impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-02-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY 1, 2, 3
),

comparison AS (
    SELECT
        client_hash_id,
        content_hash_id,
        MAX(CASE WHEN month = DATE '2026-02-01'
                 THEN impressions END) AS previous_impressions,
        MAX(CASE WHEN month = DATE '2026-03-01'
                 THEN impressions END) AS march_impressions
    FROM monthly
    GROUP BY 1, 2
)

SELECT
    CASE
        WHEN previous_impressions = 0 THEN '0'
        WHEN previous_impressions <= 10 THEN '1-10'
        WHEN previous_impressions <= 100 THEN '11-100'
        WHEN previous_impressions <= 1000 THEN '101-1000'
        ELSE '1000+'
    END AS impression_bucket,

    COUNT(*) AS n,

    ROUND(
        AVG(
            CASE
                WHEN march_impressions < 0.8 * previous_impressions
                THEN 1.0
                ELSE 0.0
            END
        ),
        3
    ) AS decline_rate

FROM comparison
WHERE previous_impressions IS NOT NULL
GROUP BY 1
ORDER BY
    CASE impression_bucket
        WHEN '0' THEN 1
        WHEN '1-10' THEN 2
        WHEN '11-100' THEN 3
        WHEN '101-1000' THEN 4
        WHEN '1000+' THEN 5
    END
""").df()

signal1_decline

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impression_bucket,n,decline_rate
0,0,167987,0.000
1,1-10,35661,0.408
2,11-100,37774,0.248
3,101-1000,46837,0.187
4,1000+,33287,0.155


### Signal #1 — Previous search demand

**Question:** Do pages with more previous search impressions show a higher rate of meaningful decline?

**Observed result:** The opposite pattern was observed. Pages with 1–10 previous 30-day impressions had a 40.8% decline rate, while pages with 1,000+ impressions had a 15.5% decline rate.

**Verdict: OPPOSITE**

The data does not support using higher search demand as a signal that a page is more likely to decline. However, volume may still be useful as a prioritization signal because a decline on a page with more search demand could represent a larger opportunity.

I will therefore avoid treating high volume as evidence of decline and will test it separately as an impact/prioritization signal.

## Signal #2 — Previous position

In [ ]:
signal2 = con.sql(f"""
WITH position_data AS (
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_avg_position) AS avg_position_30d
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-02-01'
      AND report_date < DATE '2026-03-01'
      AND gsc_data_available IS TRUE
      AND gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    CASE
        WHEN avg_position_30d <= 3 THEN '1-3'
        WHEN avg_position_30d <= 10 THEN '4-10'
        WHEN avg_position_30d <= 20 THEN '11-20'
        WHEN avg_position_30d <= 50 THEN '21-50'
        ELSE '50+'
    END AS position_bucket,

    COUNT(*) AS n,

    ROUND(AVG(avg_position_30d), 2) AS mean_position

FROM position_data

GROUP BY 1

ORDER BY
    CASE position_bucket
        WHEN '1-3' THEN 1
        WHEN '4-10' THEN 2
        WHEN '11-20' THEN 3
        WHEN '21-50' THEN 4
        WHEN '50+' THEN 5
    END
""").df()

signal2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,mean_position
0,1-3,19243,1.82
1,4-10,75898,6.32
2,11-20,31699,13.97
3,21-50,21776,30.68
4,50+,4943,65.69


In [ ]:
signal2_decline = con.sql(f"""
WITH monthly AS (
    SELECT
        client_hash_id,
        content_hash_id,
        DATE_TRUNC('month', report_date) AS month,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_impressions) AS impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-02-01'
      AND report_date < DATE '2026-04-01'
      AND gsc_data_available IS TRUE
      AND gsc_impressions > 0
    GROUP BY 1, 2, 3
),

comparison AS (
    SELECT
        client_hash_id,
        content_hash_id,

        MAX(
            CASE
                WHEN month = DATE '2026-02-01'
                THEN avg_position
            END
        ) AS previous_position,

        MAX(
            CASE
                WHEN month = DATE '2026-02-01'
                THEN impressions
            END
        ) AS previous_impressions,

        MAX(
            CASE
                WHEN month = DATE '2026-03-01'
                THEN impressions
            END
        ) AS march_impressions

    FROM monthly
    GROUP BY 1, 2
)

SELECT

    CASE
        WHEN previous_position <= 3 THEN '1-3'
        WHEN previous_position <= 10 THEN '4-10'
        WHEN previous_position <= 20 THEN '11-20'
        WHEN previous_position <= 50 THEN '21-50'
        ELSE '50+'
    END AS position_bucket,

    COUNT(*) AS n,

    ROUND(
        AVG(
            CASE
                WHEN march_impressions < 0.8 * previous_impressions
                THEN 1.0
                ELSE 0.0
            END
        ),
        3
    ) AS decline_rate

FROM comparison

WHERE previous_position IS NOT NULL
  AND previous_impressions > 0
  AND march_impressions IS NOT NULL

GROUP BY 1

ORDER BY
    CASE position_bucket
        WHEN '1-3' THEN 1
        WHEN '4-10' THEN 2
        WHEN '11-20' THEN 3
        WHEN '21-50' THEN 4
        WHEN '50+' THEN 5
    END
""").df()

signal2_decline

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,decline_rate
0,1-3,16613,0.207
1,4-10,68001,0.208
2,11-20,27609,0.217
3,21-50,18063,0.144
4,50+,3952,0.169


### Signal #2 — Search position

**Question:** Do pages in different search-position ranges show different rates of meaningful impression decline?

**Observed result:** The decline rates were similar for positions 1–3, 4–10, and 11–20, at 20.7%, 20.8%, and 21.7%. The rate then dropped to 14.4% for positions 21–50 and was 16.9% for positions 50+.

**Verdict: MIXED**

The data does not show a clear monotonic relationship between search position and impression decline. Position may still be useful as context when reviewing a page, but I would not treat it as a strong standalone decline signal.

##Signal #3 — Staleness

In [ ]:
signal3 = con.sql(f"""
WITH content_age AS (
    SELECT
        content_hash_id,
        DATE_DIFF(
            'day',
            CAST(content_created_date AS DATE),
            DATE '2026-03-01'
        ) AS age_days
    FROM {TABLES['dim_content']}
    WHERE content_created_date IS NOT NULL
),

monthly AS (
    SELECT
        client_hash_id,
        content_hash_id,
        DATE_TRUNC('month', report_date) AS month,
        SUM(gsc_impressions) AS impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-02-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY 1, 2, 3
),

comparison AS (
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        a.age_days,

        MAX(
            CASE
                WHEN m.month = DATE '2026-02-01'
                THEN m.impressions
            END
        ) AS previous_impressions,

        MAX(
            CASE
                WHEN m.month = DATE '2026-03-01'
                THEN m.impressions
            END
        ) AS march_impressions

    FROM monthly m
    JOIN content_age a
      ON m.content_hash_id = a.content_hash_id

    GROUP BY
        m.client_hash_id,
        m.content_hash_id,
        a.age_days
)

SELECT
    CASE
        WHEN age_days < 90 THEN '<90 days'
        WHEN age_days < 180 THEN '90-179 days'
        WHEN age_days < 365 THEN '180-364 days'
        ELSE '365+ days'
    END AS age_bucket,

    COUNT(*) AS n,

    ROUND(
        AVG(
            CASE
                WHEN march_impressions < 0.8 * previous_impressions
                THEN 1.0
                ELSE 0.0
            END
        ),
        3
    ) AS decline_rate

FROM comparison

WHERE age_days >= 0
  AND previous_impressions > 0
  AND march_impressions IS NOT NULL

GROUP BY 1

ORDER BY
    CASE age_bucket
        WHEN '<90 days' THEN 1
        WHEN '90-179 days' THEN 2
        WHEN '180-364 days' THEN 3
        WHEN '365+ days' THEN 4
    END
""").df()

signal3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,age_bucket,n,decline_rate
0,<90 days,42221,0.243
1,90-179 days,28043,0.276
2,180-364 days,63574,0.280
3,365+ days,11441,0.179


### Signal #3 — Content age / staleness

**Question:** Do older pages have a higher rate of meaningful impression decline?

**Observed result:** The decline rate increased from 24.3% for pages under 90 days old to 27.6% for pages aged 90–179 days and 28.0% for pages aged 180–364 days. However, the rate dropped to 17.9% for pages aged 365+ days.

**Verdict: MIXED**

The data does not show a consistent relationship where older content always has a higher decline rate. Content age may still be useful as a review signal, but it should not be treated as proof that a page needs refreshing.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Flag-linked test — Staleness behind refresh logic

I tested the assumption behind a stale visible-page review signal. I restricted the analysis to pages with at least 100 previous 30-day impressions so that the test focused on pages with meaningful search visibility.

The observed decline rate was 18.0% for pages under 180 days old, 18.3% for pages aged 180–364 days, and 18.8% for pages aged 365+ days.

**Verdict: MIXED**

The data shows a small directional increase in decline rate with age, but the difference is small. Therefore, content age can be used as a decision-support signal for review, but age alone is not enough evidence that a page needs refreshing.

In [ ]:
flag_linked_test = con.sql(f"""
WITH content_age AS (
    SELECT
        content_hash_id,
        DATE_DIFF(
            'day',
            CAST(content_created_date AS DATE),
            DATE '2026-03-01'
        ) AS age_days
    FROM {TABLES['dim_content']}
    WHERE content_created_date IS NOT NULL
),

monthly AS (
    SELECT
        client_hash_id,
        content_hash_id,
        DATE_TRUNC('month', report_date) AS month,
        SUM(gsc_impressions) AS impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-02-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY 1, 2, 3
),

comparison AS (
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        a.age_days,

        MAX(
            CASE
                WHEN m.month = DATE '2026-02-01'
                THEN m.impressions
            END
        ) AS previous_impressions,

        MAX(
            CASE
                WHEN m.month = DATE '2026-03-01'
                THEN m.impressions
            END
        ) AS march_impressions

    FROM monthly m
    JOIN content_age a
      ON m.content_hash_id = a.content_hash_id

    GROUP BY
        m.client_hash_id,
        m.content_hash_id,
        a.age_days
)

SELECT
    CASE
        WHEN age_days < 180 THEN '<180 days'
        WHEN age_days < 365 THEN '180-364 days'
        ELSE '365+ days'
    END AS age_bucket,

    COUNT(*) AS n,

    ROUND(AVG(previous_impressions), 2)
        AS avg_previous_impressions,

    ROUND(
        AVG(
            CASE
                WHEN march_impressions < 0.8 * previous_impressions
                THEN 1.0
                ELSE 0.0
            END
        ),
        3
    ) AS decline_rate

FROM comparison

WHERE age_days >= 0
  AND previous_impressions >= 100
  AND march_impressions IS NOT NULL

GROUP BY 1

ORDER BY
    CASE age_bucket
        WHEN '<180 days' THEN 1
        WHEN '180-364 days' THEN 2
        WHEN '365+ days' THEN 3
    END
""").df()

flag_linked_test

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,age_bucket,n,avg_previous_impressions,decline_rate
0,<180 days,36242,2064.97,0.180
1,180-364 days,33634,2510.82,0.183
2,365+ days,6961,2441.40,0.188


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The audit shows that no single signal is strong enough to decide that a page needs action. Search demand showed an opposite relationship with decline, while search position and content age showed mixed results.

For the content team, these signals should be used as decision-support rather than as fixed rules. Pages should be reviewed using multiple signals together, with human judgment before taking action.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.